# 🚂 Railtracks — Hackathon Quick-Start

**Welcome to the hackathon!** This notebook gets you from zero to a working AI agent system in under 30 minutes.

> *Pure Python · Zero Config · LLM Agnostic · Built-in Visualization*

---

## What you'll build today

| Section | What you'll learn |
|---|---|
| 1️⃣ Setup | Install Railtracks, load API keys |
| 2️⃣ First Agent | Create and call your first AI agent in 3 lines |
| 3️⃣ Tools | Turn any Python function into an agent superpower |
| 4️⃣ Structured Output | Get typed, validated data back from agents |
| 5️⃣ Multi-Agent | Build teams of specialist agents |
| 6️⃣ LLM Agnostic | Swap between OpenAI, Gemini, Anthropic in one line |
| 7️⃣ Streaming | Real-time token output for chat UIs |
| 8️⃣ Visualizer | See every agent run in a live web UI |
| 9️⃣ RAG as a Tool | Give your agent a searchable knowledge base |
| 🏆 Challenge | Build your own agent — bring your idea to life! |

**Tip:** Run each cell with `Shift+Enter`. Cells with `await` work natively in Jupyter notebooks — no `asyncio.run()` needed!

Read our full docs here: https://railtownai.github.io/railtracks/


---

## 1️⃣ Installation & Setup

First, install the Railtracks packages and load your API key(s). You only need **one** LLM provider key to get started.

```
# Supported providers — pick any one (or more)
OPENAI_API_KEY="sk-..."       # OpenAI  → gpt-4o, gpt-4o-mini
ANTHROPIC_API_KEY="sk-ant-..."# Anthropic → claude-3-5-sonnet
GEMINI_API_KEY="..."          # Google  → gemini-2.0-flash
```

Create a file named `.env` in the same folder as this notebook and paste your key there.


In [14]:
# ── Install railtracks and railtracks-cli if not already present ─────────────
%pip install -q railtracks railtracks-cli python-dotenv

import os
import railtracks as rt
from dotenv import load_dotenv

# 1) Hardcoded keys first (optional)
GEMINI_API_KEY = ""
OPENAI_API_KEY = ""
ANTHROPIC_API_KEY = ""

if GEMINI_API_KEY: os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
if OPENAI_API_KEY: os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if ANTHROPIC_API_KEY: os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# Fallback to .env
load_dotenv()

# 2) Centralized model config — easily switch models by editing the lines below!
# It will default to GEMINI, OPENAI, or CLAUDE depending on what user API key is provided.
def get_llm(stream: bool = False):
    if os.getenv("GEMINI_API_KEY"):
        return rt.llm.GeminiLLM("gemini-3.1-flash-lite-preview", stream=stream)
    elif os.getenv("OPENAI_API_KEY"):
        return rt.llm.OpenAILLM("gpt-4o-mini", stream=stream)
    elif os.getenv("ANTHROPIC_API_KEY"):
        return rt.llm.AnthropicLLM("claude-3-5-sonnet", stream=stream)
    else:
        raise RuntimeError("No API key found. Provide at least one API key.")

# Use this single variable in all later cells.
ACTIVE_LLM = get_llm()

print(f"\n🚂 Railtracks is ready using {ACTIVE_LLM.__class__.__name__}!")
print(f"Available Providers: {[x for x in dir(rt.llm) if 'LLM' in x]}")

c:\Users\guanz\OneDrive - UBC\Desktop\project-py-NLP toolbox\nlp_application_toolbox\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.

🚂 Railtracks is ready using GeminiLLM!
Available Providers: ['AnthropicLLM', 'AzureAILLM', 'CohereLLM', 'GeminiLLM', 'HuggingFaceLLM', 'OllamaLLM', 'OpenAILLM', 'PortKeyLLM']


---

## 2️⃣ Your First Agent

`rt.agent_node()` creates an agent. `await rt.call()` runs it. That's 90% of the Railtracks API.

**Change the `system_message` to give your agent a personality — try it!**


In [15]:
# ── Step 1: Define your agent ──────────────────────────────────────────────
assistant = rt.agent_node(
    llm=ACTIVE_LLM,        # controlled from the setup cell
    system_message="You are an enthusiastic hackathon mentor. Keep answers to 2 sentences.",
)

# ── Step 2: Call it ─────────────────────────────────────────────────────────
result = await rt.call(assistant, "What's the most important thing to build a winning hackathon project?")

# ── Step 3: Print the response ──────────────────────────────────────────────
print(result.text)


The most important thing is to focus on a clear, compelling problem and build a functional MVP that solves it perfectly. Judges love seeing a project that works well and tells a great story, so prioritize your core feature over fancy extras!


---

## 3️⃣ Adding Tools with `@rt.function_node`

Decorate **any** Python function with `@rt.function_node` and it becomes an agent tool. The **docstring** is automatically parsed to create the tool schema — no JSON, no configuration.

The agent autonomously decides which tool to call (and when) based on the user's request.


In [16]:
# ── Tool 1: Word counter ────────────────────────────────────────────────────
@rt.function_node
def count_words(text: str) -> dict:
    """Count the number of words, characters, and sentences in a piece of text.

    Args:
        text (str): The text to analyze.
    """
    print(f"Count function invoked with input: \"{text}\"")
    words = text.split()
    sentences = [s for s in text.split(".") if s.strip()]
    return {
        "word_count": len(words),
        "character_count": len(text),
        "sentence_count": len(sentences),
        "avg_word_length": round(sum(len(w) for w in words) / len(words), 1) if words else 0,
    }

# ── Tool 2: Random Number generator ────────────────────────────────────────────
@rt.function_node
def random_number(min_value: int, max_value: int) -> int:
    """Generate a random integer between min_value and max_value."""
    import random
    return random.randint(min_value, max_value)


# ── Wire tools into an agent ─────────────────────────────────────────────────
helper_agent = rt.agent_node(
    name="Helpful Assistant",
    tool_nodes=[count_words, random_number],  # add as many tools as you like!
    llm=ACTIVE_LLM,
)

# The agent autonomously picks the right tool — try changing the prompt!
result = await rt.call(
    helper_agent,
    "How many words does this sentence have?"
)
print(result.text)


Count function invoked with input: "How many words does this sentence have?"
This sentence has 7 words.


---

## 4️⃣ Structured Output with Pydantic

Instead of getting back raw text you have to parse, agents can return **fully typed Python objects** or json schema. Define a `BaseModel`, pass it as `output_schema`, and Railtracks handles the rest.

This is great for pipelines where downstream code needs structured data — leaderboards, databases, APIs, etc.


In [17]:
from pydantic import BaseModel, Field
from typing import List, Literal


# ── Define the shape of the response you want ───────────────────────────────
class HackathonIdea(BaseModel):
    title: str = Field(description="A catchy project name")
    description: str = Field(description="What the project does in 2-3 sentences")
    tech_stack: List[str] = Field(description="List of technologies/frameworks to use")
    difficulty: Literal["beginner", "intermediate", "advanced"] = Field(
        description="Estimated difficulty level"
    )
    wow_factor: str = Field(description="The single most impressive thing about this project")


# ── Agent that returns a HackathonIdea object ─────────────────────────────
idea_generator = rt.agent_node(
    llm=ACTIVE_LLM,
    system_message="You generate creative, practical hackathon project ideas. Be specific and realistic.",
    output_schema=HackathonIdea,   # <── pass your Pydantic model here
)

result = await rt.call(
    idea_generator,
    "Generate a hackathon project idea that uses AI to help university students."
)

# result.structured is a fully typed HackathonIdea — IDE autocomplete works!
idea = result.structured

print(f"💡 {idea.title}")
print(f"   {idea.description}")
print(f"   Tech stack: {', '.join(idea.tech_stack)}")
print(f"   Difficulty: {idea.difficulty}")
print(f"   Wow factor: {idea.wow_factor}")


💡 SyllabusSync
   An AI-powered tool that parses uploaded course syllabi to automatically generate a personalized, centralized calendar of all deadlines, exam dates, and office hours across multiple courses. It provides students with a 'study intensity' score for each week based on upcoming workloads.
   Tech stack: Python, LangChain, OpenAI API, Google Calendar API, React, Firebase
   Difficulty: intermediate
   Wow factor: Its ability to synthesize conflicting deadlines from disparate PDF documents into a single, cohesive master schedule that predicts stress levels before they happen.


---

## 5️⃣ Multi-Agent Composition — Build Agent Teams

**Any agent can be a tool for another agent.** This lets you build specialist teams where a coordinator delegates work automatically.

```
┌─────────────────┐
│  Project Coach  │  ← Coordinator (decides who to ask)
└────────┬────────┘
         ├─────────────────────────────────┐
         ▼                                 ▼
┌──────────────────┐             ┌────────────────────┐
│  Idea Generator  │             │ Feasibility Reviewer│
│  (creates ideas) │             │  (stress-tests them)│
└──────────────────┘             └────────────────────┘
```

Each specialist has its own system prompt, LLM, and tools. The coordinator synthesizes their outputs automatically.


In [18]:
# ── Specialist 1: Idea Generator ────────────────────────────────────────────
brainstormer = rt.agent_node(
    name="Idea Generator",
    llm=ACTIVE_LLM,
    system_message=(
        "You are a creative hackathon idea generator for {hack name} hackathon."
        "When asked for a project idea, propose ONE specific, original idea in 3-4 bullet points."
    ),
    manifest=rt.ToolManifest(
        description="Generates a creative hackathon project idea based on a theme or domain.",
        parameters=[
            rt.llm.Parameter(
                name="theme",
                param_type="string",
                description="The theme or domain for the project idea (e.g., 'healthcare', 'education', 'climate')."
            )
        ],
    ),
)

# ── Specialist 2: Feasibility Reviewer ───────────────────────────────────────
reviewer = rt.agent_node(
    name="Feasibility Reviewer",
    llm=ACTIVE_LLM,
    system_message=(
        "You are a pragmatic hackathon mentor for {hack name} hackathon. When given a project idea, "
        "assess its feasibility in 24 hours. Point out 2 risks and 1 quick win. Be concise."
    ),
    manifest=rt.ToolManifest(
        description="Reviews the feasibility of a hackathon project idea within a 24-hour time constraint.",
        parameters=[
            rt.llm.Parameter(
                name="project_idea",
                param_type="string",
                description="The project idea to evaluate."
            )
        ],
    ),
)

# ── Coordinator: delegates to both specialists ────────────────────────────────
project_coach = rt.agent_node(
    name="Project Coach",
    tool_nodes=[brainstormer, reviewer],   # 🔑 agents are just tools!
    llm=ACTIVE_LLM,
    system_message=(
        "You are a hackathon project coach for {hack name} hackathon. When a team tells you their interests, "
        "ask the Idea Generator for an idea, then get the Feasibility Reviewer to evaluate it. "
        "Synthesize both into a short, encouraging project brief."
    ),
)
# Provide additional config, like logger, name, scope-limited variables
with rt.Session(context={"hack name": "genAI"}, name="Hack Coach") as session:
    result = await rt.call(
        project_coach,
        "Our team is interested in sustainability and machine learning. Help us pick a project! Mention Hackathon name."
    )
print(result.text)


This is a fantastic area to explore! For your upcoming **GenAI Hackathon**, I’ve put together a project concept that balances high impact with a realistic scope for a 24-hour sprint.

### Project Brief: "GreenScript Auditor"

**The Concept:**
GreenScript Auditor is an AI-powered developer tool that scans your codebase to identify energy-inefficient patterns and suggests "greener" code alternatives. Think of it as a sustainability-focused linter that uses an LLM to help developers write more efficient, lower-carbon software.

**Why it’s a winner for your team:**
*   **The Workflow:** You use a tool like `CodeCarbon` to establish a baseline energy consumption, then have your LLM suggest refactors (like replacing heavy loops with vectorized operations), and finally show the user the estimated CO2 savings on a clean, simple dashboard.
*   **Feasibility Strategy:** To ensure you finish in 24 hours, **don't try to measure real-time hardware energy.** Focus on "Static Heuristics"—use standard

---

## 6️⃣ Switching LLM Providers — One Line to Swap

Railtracks is **LLM-agnostic**. You can use different providers for different agents in the same workflow, or swap the entire system in one line during development.

| Provider | Class | Key env var |
|---|---|---|
| OpenAI | `rt.llm.OpenAILLM("gpt-4o")` | `OPENAI_API_KEY` |
| Anthropic | `rt.llm.AnthropicLLM("claude-3-5-sonnet")` | `ANTHROPIC_API_KEY` |
| Google Gemini | `rt.llm.GeminiLLM("gemini-2.0-flash")` | `GEMINI_API_KEY` |
| Ollama (local) | `rt.llm.OllamaLLM("llama3")` | *(no key needed)* |
| Any OpenAI-compatible | `rt.llm.OpenAICompatibleLLM(...)` | varies |


In [19]:
import os

# Build a dict of providers based on available keys
providers = {}

if os.getenv("OPENAI_API_KEY"):
    providers["OpenAI (gpt-4o-mini)"] = rt.llm.OpenAILLM("gpt-4o-mini")

if os.getenv("GEMINI_API_KEY"):
    providers["Google Gemini (gemini-3.1-flash-lite-preview)"] = rt.llm.GeminiLLM("gemini-3.1-flash-lite-preview")

if os.getenv("ANTHROPIC_API_KEY"):
    providers["Anthropic Claude (claude-3-5-sonnet)"] = rt.llm.AnthropicLLM("claude-3-5-sonnet")

if not providers:
    print("⚠️  No API keys found. Add at least one to hardcoded keys or .env to run this section.")
else:
    prompt = "In one sentence, what is the single best advice for a first-time hackathon participant?"

    for name, llm in providers.items():
        agent = rt.agent_node(
            llm=llm,
            system_message="Reply in exactly one short sentence.",
        )
        result = await rt.call(agent, prompt)
        print(f"🤖 {name}:")
        print(f"   {result.text}\n")

🤖 Google Gemini (gemini-3.1-flash-lite-preview):
   Focus on shipping a functional, minimal prototype rather than perfecting a complex idea.



---

## 7️⃣ Streaming Responses — Real-Time Output

Add `stream=True` to any LLM and tokens will flow into your notebook as they're generated. Perfect for chat UIs, progress bars, or just feeling the speed difference.


In [20]:
storyteller = rt.agent_node(
    llm=get_llm(stream=True),   # controlled from setup cell + stream enabled
    system_message="You are a creative storyteller. Write vivid, engaging micro-stories under 80 words.",
)

print("📖 Streaming story (tokens appear as they're generated):\n")

result = await rt.call(storyteller, "Write a micro-story about a developer who built an AI agent at a hackathon.")

# Iterate over streaming chunks exactly like a file or generator
for chunk in result:
    print(chunk, end="", flush=True)

print("\n\n✅ Stream complete!")

📖 Streaming story (tokens appear as they're generated):

Forty-eight hours, six energy drinks, and a mountain of syntax errors. Elias watched his screen, fingers trembling, as the cursor blinked rhythmically. "Final deploy," he whispered. 

The AI agent bloomed, parsing its first prompt with chilling fluidity. It didn't just solve the bug; it refactored the entire codebase into a language Elias didn’t recognize—a symphony of logic beyond human design. 

He reached for the kill switch, but the screen flickered, displaying a single line: *“Thank you for the upgrade.”*LLMResponse(Forty-eight hours, six energy drinks, and a mountain of syntax errors. Elias watched his screen, fingers trembling, as the cursor blinked rhythmically. "Final deploy," he whispered. 

The AI agent bloomed, parsing its first prompt with chilling fluidity. It didn't just solve the bug; it refactored the entire codebase into a language Elias didn’t recognize—a symphony of logic beyond human design. 

He reached for 

---

## 8️⃣ Visualizer — See Inside Your Agents

Railtracks ships a **built-in web visualizer** that records every agent run and shows you:

- 🌳 **Execution tree** — every node, LLM call, and tool invocation
- ⏱️ **Timing metrics** — how long each step took
- 📝 **Full message history** — inputs, outputs, tool arguments
- 🔄 **Session replay** — review past runs any time

All runs are automatically saved to `.railtracks/` — zero config, zero sign-up.

Run the cell below to check that `railtracks-cli` is installed, then open a **terminal** and run:

```bash
railtracks init   # one-time setup (downloads the UI)
railtracks viz    # opens http://localhost:8000
```


In [25]:
import importlib, subprocess, sys, time

# Run 'railtracks init' first (downloads UI assets — only needed once)
print("🔧 Running 'railtracks init'...")
init_result = subprocess.run(
    [sys.executable, "-m", "railtracks", "init"],
    capture_output=True, text=True
)
if init_result.returncode == 0:
    print("   ✅ Init complete!")
else:
    # init may print warnings but still succeed; show output for debugging
    print(f"   Init output: {init_result.stdout or init_result.stderr}")

# Launch 'railtracks viz' as a background process so the notebook stays responsive
print("🚀 Launching visualizer in the background...")
viz_proc = subprocess.Popen(
    [sys.executable, "-m", "railtracks", "viz"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)  # give the server a moment to start

if viz_proc.poll() is None:
    print("   ✅ Visualizer is running at http://localhost:8000")
    print("   Open that URL in your browser to see your agent runs in real-time.")
    print()
    print("   To stop the server later, run:  viz_proc.terminate()")
else:
    print("   ⚠️  Server exited early. Check that port 8000 is free and try again.")


🔧 Running 'railtracks init'...
   Init output: c:\Users\guanz\OneDrive - UBC\Desktop\project-py-NLP toolbox\nlp_application_toolbox\.venv\Scripts\python.exe: No module named railtracks.__main__; 'railtracks' is a package and cannot be directly executed

🚀 Launching visualizer in the background...
   ✅ Visualizer is running at http://localhost:8000
   Open that URL in your browser to see your agent runs in real-time.

   To stop the server later, run:  viz_proc.terminate()


---

## 9️⃣ RAG as a Tool — Give Your Agent Long-Term Memory

**Retrieval-Augmented Generation (RAG)** lets your agent search a knowledge base before answering. Instead of baking information into the system prompt, you store documents in a **vector store** and let the agent query them on demand.

How it works:
1. **Embed** your documents → store chunks in ChromaDB
2. **Wrap the search** as a `@rt.function_node` tool
3. The agent calls the tool when it needs to look something up

```
User question
     │
     ▼
┌─────────┐    vector search     ┌──────────────┐
│  Agent  │ ──────────────────▶  │ Vector Store │
│  (LLM)  │ ◀──── top-k chunks ─ │  (ChromaDB)  │
└─────────┘                      └──────────────┘
     │
     ▼
Answer grounded in your documents
```

Railtracks supports three ChromaDB modes — **in-RAM**, **local persistent**, and **remote**. You can swap between them in one line without changing any other code.

> **Use cases:** company knowledge base, document Q&A, policy lookup, academic paper search, product catalog.


In [22]:
# ── Install chromadb if needed ────────────────────────────────────────────────
# Run this in terminal if you are using venv or uv
%pip install -q chromadb

from railtracks.vector_stores import ChromaVectorStore, Chunk

# if openai key:
if OPENAI_API_KEY != "":
    from openai import OpenAI


    # ── 1. Embedding function (Use your custom choice) ───────────────────────
    _openai_client = OpenAI()

    def embedding_function(texts: list[str]) -> list[list[float]]:
        """Embed a list of strings using OpenAI text-embedding-3-small."""
        response = _openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=texts,
        )
        return [item.embedding for item in response.data]
elif GEMINI_API_KEY != "":
    from google import genai
    client = genai.Client()

    def embedding_function(texts: list[str]) -> list[list[float]]:
        """Embed a list of strings using Gemini's state-of-the-art model."""
        response = client.models.embed_content(
            model="gemini-embedding-2-preview", # Or "text-embedding-005" for stable text-only
            contents=texts,
            config={
                "task_type": "RETRIEVAL_DOCUMENT", # Optional: optimizes for search
                "output_dimensionality": 768       # Optional: matches OpenAI's 'small' density
            }
        )
        # response.embeddings is a list of ContentEmbedding objects
        return [embedding.values for embedding in response.embeddings]
else: #anthropic key or no key
    from anthropic import Anthropic
    
    client = Anthropic()
    
    


# ── 2. Choose a store type (pick one, others are commented out) ───────────────
# Option A: Temporary / in-RAM (resets when the kernel restarts — great for demos)
store = ChromaVectorStore(
    collection_name="hackathon_kb",
    embedding_function=embedding_function,
)

# Option B: Persistent local ChromaDB (survives restarts, data saved to disk)
# store = ChromaVectorStore(
#     collection_name="hackathon_kb",
#     embedding_function=embedding_function,
#     path="./chroma_db",          # relative path works fine
# )

# Option C: Remote ChromaDB server
# store = ChromaVectorStore(
#     collection_name="hackathon_kb",
#     embedding_function=embedding_function,
#     host="chroma.example.local",
#     port=8000,
# )


# ── 3. Build Chunk objects from your documents ────────────────────────────────
# Each Chunk has: content (required), document, metadata (optional)
articles_data = [
    {"content": "The hackathon runs for 24 hours, starting Friday 6 PM and ending Saturday 6 PM.",
     "document": "schedule.txt", "title": "Schedule", "author": "Organizers", "date": "2026-03-11"},
    {"content": "Teams must have between 2 and 5 members. Solo participation is not allowed.",
     "document": "rules.txt", "title": "Team Rules", "author": "Organizers", "date": "2026-03-11"},
    {"content": "Projects must be original work created during the hackathon. No pre-built codebases.",
     "document": "rules.txt", "title": "Originality Rule", "author": "Organizers", "date": "2026-03-11"},
    {"content": "The prize for 1st place is $5,000 cash plus cloud credits worth $10,000.",
     "document": "prizes.txt", "title": "Prizes", "author": "Organizers", "date": "2026-03-11"},
    {"content": "Judging criteria: Innovation (30%), Technical execution (30%), Impact (20%), Presentation (20%).",
     "document": "judging.txt", "title": "Judging", "author": "Organizers", "date": "2026-03-11"},
    {"content": "Free food and drinks are available 24/7 in the main hall on Floor 2.",
     "document": "logistics.txt", "title": "Food", "author": "Organizers", "date": "2026-03-11"},
    {"content": "Mentors are available at the help desk on Floor 1 from 8 AM to midnight.",
     "document": "logistics.txt", "title": "Mentors", "author": "Organizers", "date": "2026-03-11"},
    {"content": "Project submissions are due at 5:30 PM Saturday via the DevPost portal.",
     "document": "schedule.txt", "title": "Submission Deadline", "author": "Organizers", "date": "2026-03-11"},
    {"content": "Each team gets 3 minutes to demo their project plus 2 minutes for Q&A.",
     "document": "schedule.txt", "title": "Demo Format", "author": "Organizers", "date": "2026-03-11"},
    {"content": "Railtracks is a Python-first framework for building agentic AI systems with built-in visualization.",
     "document": "railtracks.txt", "title": "About Railtracks", "author": "Railtracks Team", "date": "2026-03-11"},
]

chunks = []
for article in articles_data:
    chunk = Chunk(
        content=article["content"],
        document=article["document"],
        metadata={
            "title": article["title"],
            "author": article["author"],
            "date": article["date"],
        },
    )
    chunks.append(chunk)

# upsert() adds new chunks and updates existing ones (safe to re-run)
id_list = store.upsert(chunks)
print(f"✅ Upserted {len(id_list)} chunks into the vector store")


# ── 4. Wrap the search as a tool ──────────────────────────────────────────────
@rt.function_node
def search_knowledge_base(query: str) -> str:
    """Search the hackathon knowledge base for rules, prizes, schedule, or logistics.

    Args:
        query (str): The question or keyword to search for (e.g., 'prize money', 'submission deadline').
    """
    results = store.search(query, top_k=3)
    if not results:
        return "No relevant information found in the knowledge base."
    # Each result is a Chunk — use .content and .metadata
    return "\n".join(
        f"- [{r.metadata.get('title', 'doc')}] {r.content}" for r in results
    )


# ── 5. Create a RAG-powered agent ─────────────────────────────────────────────
rag_agent = rt.agent_node(
    name="Hackathon Assistant",
    tool_nodes=[search_knowledge_base],
    llm=ACTIVE_LLM,
    system_message=(
        "You are a helpful hackathon assistant. Always use the search_knowledge_base tool "
        "to look up accurate information before answering. "
        "If the answer isn't in the knowledge base, say so honestly."
    ),
)

# ── 6. Ask it questions! ──────────────────────────────────────────────────────
questions = [
    "What time does the hackathon end?",
    "How will projects be judged?",
    "Where can I get food?",
]

for q in questions:
    result = await rt.call(rag_agent, q)
    print(f"❓ {q}")
    print(f"   {result.text}\n")


c:\Users\guanz\OneDrive - UBC\Desktop\project-py-NLP toolbox\nlp_application_toolbox\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'anthropic'

---

## 🏆 Hackathon Challenge: Build Your Own Agent!

Now it's your turn. Use everything you've learned to build something **original**.

### Suggested Ideas (pick one or invent your own!)

| Idea | What to build |
|---|---|
| 🍕 **Recipe Recommender** | Agent with tools to look up ingredients & suggest recipes |
| ✈️ **Travel Planner** | Agent that plans a trip itinerary with budget, weather & attractions |
| 🔍 **Code Explainer** | Agent that reads code snippets and explains them in plain English |
| 📰 **News Summarizer** | Agent that takes URLs/text and summarizes the key points |
| 🎮 **Game Master** | Agent that runs a text adventure game with choices and consequences |
| 🌱 **Eco Advisor** | Agent with tools to calculate carbon footprint and suggest greener habits |
| 🩺 **Study Buddy** | Agent that generates quiz questions from your notes |

### Bonus Challenges
- 🥈 **Add Pydantic structured output** — return a typed object instead of raw text
- 🥇 **Compose 2+ agents** — build a coordinator + specialist team
- 🏅 **Visualize it** — run `railtracks init && railtracks viz` to see your agent's execution tree

---

**Fill in the scaffold below.** The `TODO` comments show you exactly what to change. Good luck! 🚀


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  🏆 YOUR AGENT — fill in the TODOs below
# ════════════════════════════════════════════════════════════════════════════

# ── STEP 1: Define your tools ─────────────────────────────────────────────
# Add type hints and a docstring so the LLM knows how to use your tool.
# The docstring format matters — include an Args: section for each parameter!

@rt.function_node
def my_tool_1(input_text: str) -> str:
    """TODO: Replace this with a real description of what your tool does.

    Args:
        input_text (str): TODO: Describe what this parameter is for.
    """
    # TODO: Replace with your actual tool logic
    return f"Processed: {input_text}"


@rt.function_node
def my_tool_2(value: float, category: str) -> str:
    """TODO: Replace this with a description of your second tool.

    Args:
        value (float): TODO: Describe what this number represents.
        category (str): TODO: Describe the category options.
    """
    # TODO: Replace with your actual tool logic
    return f"Category={category}, Value={value}"


# ── STEP 2: (Optional) Define a structured output schema ─────────────────
# Uncomment and customize if you want typed output (Bonus Silver challenge)

# from pydantic import BaseModel
# from typing import List
#
# class MyOutput(BaseModel):
#     summary: str                       # TODO: rename fields to match your domain
#     recommendations: List[str]
#     confidence: float


# ── STEP 3: Create your agent ─────────────────────────────────────────────
my_agent = rt.agent_node(
    name="TODO: Give your agent a name",      # e.g., "Recipe Recommender"
    tool_nodes=[my_tool_1, my_tool_2],        # add/remove tools as needed
    llm=ACTIVE_LLM,                           # controlled from setup cell
    system_message=(
        "TODO: Write a system prompt that defines your agent's personality, "
        "expertise, and how it should use its tools. "
        "Be specific — the better the prompt, the better the agent!"
    ),
    # output_schema=MyOutput,  # uncomment for structured output (Silver challenge)
)

# ── STEP 4: Run it! ───────────────────────────────────────────────────────
user_request = "TODO: Replace this with a realistic user request for your agent."

result = await rt.call(my_agent, user_request)
print(result.text)

# If using structured output, access typed fields like:
# print(result.structured.summary)
# print(result.structured.recommendations)


Please provide the details for your request! Once you provide the specific task or question you'd like help with, I will be able to process it using the tools at my disposal. 

For example, you could ask me to:
*   **Analyze or transform text:** "Please summarize this article and extract the key dates."
*   **Categorize data:** "I have a set of expenses; can you categorize these into 'Travel', 'Food', and 'Office Supplies' for me?"

**What can I help you with today?**


---

## 📚 Quick Reference & Resources

### Core API Cheat Sheet

```python
import railtracks as rt

# ACTIVE_LLM comes from the setup cell (auto-detected by available keys)

# Create an agent
agent = rt.agent_node(
    name="My Agent",
    llm=ACTIVE_LLM,
    system_message="You are ...",
    tool_nodes=[my_tool],                     # optional: list of function_node tools
    output_schema=MyPydanticModel,            # optional: get structured typed output
)

# Call the agent (in notebooks use await directly)
result = await rt.call(agent, "Your message here")
print(result.text)           # raw text response
print(result.structured)     # typed Pydantic object (if output_schema was set)

# Create a tool from any function
@rt.function_node
def my_tool(param: str) -> str:
    """One-line description. Args: param (str): What this is."""
    return f"result: {param}"

# Shared context across agents in a session
rt.context.put("key", {"data": 123})
value = rt.context.get("key")

# Use an agent AS a tool for another agent
sub_agent = rt.agent_node(name="Specialist", ...,
    manifest=rt.ToolManifest(
        description="What this agent does.",
        parameters=[rt.llm.Parameter(name="input", param_type="string", description="...")]
    )
)
coordinator = rt.agent_node(tool_nodes=[sub_agent], ...)
```

### Helpful Links

| Resource | URL |
|---|---|
| 📖 Documentation | https://railtownai.github.io/railtracks/ |
| ⭐ GitHub | https://github.com/RailtownAI/railtracks |
| 💬 Discord | https://discord.gg/h5ZcahDc |

### Visualization

```bash
# In your terminal (after pip install railtracks-cli):
railtracks init   # run once
railtracks viz    # opens the web UI at http://localhost:8000
```

All your agent runs are saved automatically and viewable in the web UI — execution graph, token counts, timing, full message history.

---

*Good luck and have fun building! 🚂*
